In [1]:
# Import NLTK library for natural language processing tasks
# Импорт библиотеки NLTK для задач обработки естественного языка
import nltk

# Download the stopwords dataset from NLTK (common words like 'a', 'the', 'and')
# Скачивание набора стоп-слов из NLTK (частые слова: 'a', 'the', 'and')
nltk.download('stopwords')

# Import stopwords list to filter out common meaningless words
# Импорт списка стоп-слов для фильтрации частых, но малозначимых слов
from nltk.corpus import stopwords

# Import word_tokenize function to split text into individual words
# Импорт функции токенизации для разбиения текста на отдельные слова
from nltk import word_tokenize

# Import WordNetLemmatizer to convert words to their base/dictionary form
# Импорт лемматизатора для приведения слов к словарной форме
from nltk.stem import WordNetLemmatizer

# Import TfidfVectorizer to create TF-IDF representation (weighted word frequencies)
# Импорт TF-IDF векторизатора для создания взвешенного представления слов
from sklearn.feature_extraction.text import TfidfVectorizer

# Import CountVectorizer to create Bag-of-Words representation (simple word counts)
# Импорт CountVectorizer для создания представления "мешок слов" (простой подсчет)
from sklearn.feature_extraction.text import CountVectorizer

# Import fetch_20newsgroups to load the 20 Newsgroups dataset
# Импорт функции загрузки датасета 20 Newsgroups
from sklearn.datasets import fetch_20newsgroups

# Import re module for regular expressions (pattern matching in text)
# Импорт модуля re для регулярных выражений (поиск шаблонов в тексте)
import re

# Import string module to access punctuation and printable characters
# Импорт модуля string для доступа к знакам препинания и печатным символам
import string

# Import pandas for data manipulation and DataFrame operations
# Импорт pandas для манипуляции данными и операций с DataFrame
import pandas as pd

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\V1kor\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
%%time
# Load the 20 Newsgroups dataset (training subset)
# This downloads ~60MB of text data (11,000+ documents)
# Загрузка датасета 20 Newsgroups (обучающая выборка)
# Скачивается ~60MB текстовых данных (11,000+ документов)
newsgroups_data_sample = fetch_20newsgroups(subset='train')

# ALTERNATIVE (more efficient): Load only specific categories
# newsgroups_data_sample = fetch_20newsgroups(
#     subset='train',
#     remove=('headers', 'footers', 'quotes'),  # Remove extra metadata
#     categories=['alt.atheism', 'soc.religion.christian']  # Only 2 categories
# )

CPU times: total: 375 ms
Wall time: 406 ms


In [3]:
# Create WordNet Lemmatizer object for converting words to base form
# Создание объекта лемматизатора WordNet для приведения слов к базовой форме
lemmatizer = WordNetLemmatizer()

# Convert the loaded text data into a pandas DataFrame for easier manipulation
# Преобразование загруженных текстовых данных в pandas DataFrame для удобной работы
newsgroups_text_df = pd.DataFrame({'text': newsgroups_data_sample['data']})

# Display first 5 rows to verify data loaded correctly
# Вывод первых 5 строк для проверки корректности загрузки
newsgroups_text_df.head()

,text
0,From: lerxst@wam.umd.edu (where's my thing)\nS...
1,From: guykuo@carson.u.washington.edu (Guy Kuo)...
2,From: twillis@ec.ecn.purdue.edu (Thomas E Will...
3,From: jgreen@amber (Joe Green)\nSubject: Re: W...
4,From: jcm@head-cfa.harvard.edu (Jonathan McDow...


In [4]:
# Check the shape (number of rows and columns) of the DataFrame
# Проверка размера (количество строк и столбцов) DataFrame
# Output: (11314, 1) - 11,314 documents
newsgroups_text_df.shape

(11314, 1)

In [5]:
def clean_text(text):
    """
    Clean and preprocess text data.
    Steps: 1. Remove special chars  2. Tokenize  3. Remove stopwords  4. Lemmatize
    Очистка и предобработка текстовых данных.
    Шаги: 1. Удаление спецсимволов  2. Токенизация  3. Удаление стоп-слов  4. Лемматизация
    """
    # Convert to string and lowercase / Преобразование в строку и нижний регистр
    text = str(text).lower()
    
    # Remove special characters (keep only letters, numbers, and spaces)
    # Удаление спецсимволов (оставляем только буквы, цифры и пробелы)
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    
    # Tokenize into words / Токенизация на слова
    tokens = word_tokenize(text)
    
    # Remove stopwords and lemmatize / Удаление стоп-слов и лемматизация
    cleaned_tokens = []
    for token in tokens:
        if token not in stop_words and len(token) > 1:  # Also remove single characters
            # Lemmatize and add to list / Лемматизация и добавление в список
            cleaned_tokens.append(lemmatizer.lemmatize(token))
    
    # Join tokens back into a single string / Объединение токенов обратно в строку
    return ' '.join(cleaned_tokens)

In [6]:
%%time
# Get the default English stopwords list from NLTK
# Получение стандартного списка английских стоп-слов из NLTK
stop_words = stopwords.words('english')

# Add all printable characters (punctuation, digits, whitespace) to stop words
# This ensures all special characters are removed during cleaning
# Добавление всех печатных символов (знаки препинания, цифры, пробелы) в стоп-слова
# Это гарантирует удаление всех специальных символов при очистке
stop_words = stop_words + list(string.printable)

# Apply the cleaning function to all documents
# Применение функции очистки ко всем документам
newsgroups_text_df['cleaned_text'] = newsgroups_text_df['text'].apply(clean_text)

CPU times: total: 1min 3s
Wall time: 1min 6s


In [7]:
# Create Bag-of-Words model with maximum 20 features (most frequent words)
# Создание модели "мешок слов" с максимум 20 признаками (самые частые слова)
bag_of_words_model = CountVectorizer(max_features=20)

# Fit the model and transform the cleaned text into a document-term matrix
# Обучение модели и преобразование очищенного текста в матрицу "документ-термин"
# .todense() converts sparse matrix to dense numpy array
# .todense() преобразует разреженную матрицу в плотный numpy массив
bow_matrix = bag_of_words_model.fit_transform(newsgroups_text_df['cleaned_text']).todense()

# Convert to DataFrame for better visualization
# Преобразование в DataFrame для лучшей визуализации
bag_of_word_df = pd.DataFrame(bow_matrix)

# Set column names as the sorted vocabulary (the actual words)
# Установка имен столбцов как отсортированный словарь (фактические слова)
bag_of_word_df.columns = sorted(bag_of_words_model.vocabulary_)

# Display first 5 rows of BoW matrix
# Вывод первых 5 строк BoW матрицы
print("Bag-of-Words Matrix (first 5 documents):")
bag_of_word_df.head()

Bag-of-Words Matrix (first 5 documents):


,article,ax,com,edu,get,host,know,like,line,max,nntp,one,organization,people,posting,subject,time,university,would,writes
0,0,0,0,2,0,1,1,0,1,0,1,0,1,0,1,1,0,1,0,0
1,1,0,0,3,0,1,0,0,1,0,1,0,1,0,1,1,0,1,0,0
2,0,0,0,2,1,0,1,1,2,0,0,1,1,1,0,1,1,1,0,0
3,1,0,2,2,1,1,1,1,1,0,1,0,1,0,1,1,0,0,0,1
4,2,0,2,3,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,1


In [8]:
# Find top 10 terms in BoW (across all documents)
# Поиск топ-10 терминов в BoW (по всем документам)
print("\n" + "="*60)
print("TOP 10 TERMS IN BAG-OF-WORDS:")
print("="*60)
bow_term_frequencies = bag_of_word_df.sum().sort_values(ascending=False)
print(bow_term_frequencies.head(10))


TOP 10 TERMS IN BAG-OF-WORDS:
ax              62451
edu             21321
line            13184
subject         12334
com             12134
organization    11415
one              9485
would            8910
writes           7844
article          7700
dtype: int64


In [9]:
# Create TF-IDF model with maximum 20 features
# Создание TF-IDF модели с максимум 20 признаками
# max_features=20 keeps only the 20 most important terms
# max_features=20 оставляет только 20 самых важных терминов
tfidf_model = TfidfVectorizer(max_features=20)

# Fit and transform the cleaned text into TF-IDF matrix
# Обучение и преобразование очищенного текста в TF-IDF матрицу
tfidf_matrix = tfidf_model.fit_transform(newsgroups_text_df['cleaned_text']).todense()

# Convert to DataFrame
# Преобразование в DataFrame
tfidf_df = pd.DataFrame(tfidf_matrix)

# Set column names as sorted vocabulary
# Установка имен столбцов как отсортированный словарь
tfidf_df.columns = sorted(tfidf_model.vocabulary_)

# Display first 5 rows of TF-IDF matrix
# Вывод первых 5 строк TF-IDF матрицы
print("\nTF-IDF Matrix (first 5 documents):")
tfidf_df.head()


TF-IDF Matrix (first 5 documents):


,article,ax,com,edu,get,host,know,like,line,max,nntp,one,organization,people,posting,subject,time,university,would,writes
0,0.000000,0.0,0.000000,0.522408,0.000000,0.338589,0.399935,0.000000,0.183761,0.0,0.341216,0.000000,0.190375,0.00000,0.327584,0.183242,0.000000,0.353795,0.0,0.000000
1,0.282739,0.0,0.000000,0.691591,0.000000,0.298827,0.000000,0.000000,0.162181,0.0,0.301146,0.000000,0.168019,0.00000,0.289115,0.161723,0.000000,0.312248,0.0,0.000000
2,0.000000,0.0,0.000000,0.411404,0.325661,0.000000,0.314954,0.308844,0.289428,0.0,0.000000,0.277716,0.149923,0.35893,0.000000,0.144306,0.345624,0.278619,0.0,0.000000
3,0.234838,0.0,0.499523,0.382949,0.303137,0.248201,0.293170,0.287482,0.134705,0.0,0.250127,0.000000,0.139553,0.00000,0.240134,0.134325,0.000000,0.000000,0.0,0.225159
4,0.493319,0.0,0.524670,0.603340,0.000000,0.000000,0.000000,0.000000,0.141486,0.0,0.000000,0.000000,0.146579,0.00000,0.000000,0.141087,0.000000,0.000000,0.0,0.236494


In [10]:
# Find top 10 terms in TF-IDF (average across all documents)
# Поиск топ-10 терминов в TF-IDF (среднее по всем документам)
print("\n" + "="*60)
print("TOP 10 TERMS IN TF-IDF:")
print("="*60)
tfidf_term_scores = tfidf_df.mean().sort_values(ascending=False)
print(tfidf_term_scores.head(10))


TOP 10 TERMS IN TF-IDF:
edu             0.309661
com             0.218808
line            0.170357
subject         0.161100
organization    0.156865
one             0.142371
would           0.141189
writes          0.128910
university      0.128077
article         0.126362
dtype: float64
